# DARTS-Inspired Differentiable Search on CIFAR-10

Run this notebook in Google Colab with `Runtime > Change runtime type > GPU`.

The search settings live in `experiments/darts_search.yaml`. The implementation lives in `models/darts_model.py` and `nas/darts_search.py`, launched by `scripts/run_darts_search.py`.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/iwadas/GGSN-project.git"
BRANCH = "main"
REPO_DIR = Path("/content/GGSN-project")

if not Path("pyproject.toml").exists():
    os.chdir("/content")
    if not REPO_DIR.exists():
        !git clone -b {BRANCH} {REPO_URL}
    os.chdir(REPO_DIR)

print("Working directory:", Path.cwd())

In [ ]:
%pip install -q uv
!uv pip install --system -q optuna numpy pandas matplotlib pyyaml tqdm

In [ ]:
import torch

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

## Review Search Config

In [ ]:
!cat experiments/darts_search.yaml

## Run DARTS Differentiable Search

This runs three phases:
1. **Differentiable search** — trains network weights and architecture α parameters
2. **Derive architecture** — picks the highest-weighted operation per layer
3. **Retrain** — trains the discrete architecture from scratch and evaluates on test set

In [ ]:
!python scripts/run_darts_search.py --config experiments/darts_search.yaml

## Show Results

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

summary = json.loads(Path("results/darts_summary.json").read_text())
display(summary)

alpha_log = pd.read_csv("results/darts_alpha_log.csv")
display(alpha_log)

display(Image("plots/darts_alpha_convergence.png"))
display(Image("plots/darts_best_training_curves.png"))

## Compare with Evolutionary NAS

If you have run the evolutionary NAS notebook first, the DARTS summary will include
a `comparison_with_evolutionary_nas` section with accuracy, parameter, and latency deltas.

In [ ]:
summary = json.loads(Path("results/darts_summary.json").read_text())
comparison = summary.get("comparison_with_evolutionary_nas")
if comparison:
    display(comparison)
else:
    print("No evolutionary NAS results found. Run the evolutionary NAS notebook first.")

## Files To Keep

- `results/darts_alpha_log.csv`
- `results/darts_derived_genome.json`
- `results/darts_summary.json`
- `results/darts_best_training_log.csv`
- `plots/darts_alpha_convergence.png`
- `plots/darts_best_training_curves.png`

The checkpoint `checkpoints/darts_best_cnn.pt` is useful for resuming/evaluation.

In [ ]:
!ls -lh results plots checkpoints